## Dependencies 

In [5]:
!pip install -q transformers datasets accelerate peft bitsandbytes trl
!pip install -q evaluate bert-score rouge-score nltk sentence-transformers
!pip install -q huggingface-hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.8/838.8 kB 37.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible

# Libraries

In [6]:
import json
import torch
import numpy as np
import re
import os
import zipfile
from pathlib import Path
from typing import List, Dict, Any

from datasets import Dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM, 
    BitsAndBytesConfig, 
    TrainingArguments, 
    Trainer,
    pipeline
)
from peft import LoraConfig, get_peft_model, PeftModel
import evaluate
from bert_score import BERTScorer
from nltk.translate.meteor_score import meteor_score
import nltk
from sentence_transformers import SentenceTransformer, util
from tqdm import tqdm

nltk.download('wordnet', quiet=True)
nltk.download('punkt', quiet=True)

True

# Load Dataset

In [7]:
with open("/kaggle/input/datasets/quiz_dataset.json", "r") as f:
    data = json.load(f)

print(f"Loaded {len(data)} examples")
print("\nSample entry:")
print(json.dumps(data[0], indent=2))

Loaded 8042 examples

Sample entry:
{
  "instruction": "Generate 8 mcq questions",
  "course": "Cloud_Computing",
  "chunk_index": 0,
  "mode": "mcq",
  "n_questions": 8,
  "input": "Copyright \u00a9 2019 Huawei Technologies Co., Ltd. All rights reserved. A Brief Introduction to Cloud Computing Page 1 Copyright \u00a9 2019 Huawei Technologies Co., Ltd. All rights reserved. Foreword \u26ab IT is a fast-changing industry. Cloud computing has been developing rapidly in recent years and has become the foundation of a wide range of major applications. So, what is cloud computing? How has it evolved to what it is today? This chapter will offer you a brief introduction to the history and present of cloud computing. Page 2 Copyright \u00a9 2019 Huawei Technologies Co., Ltd. All rights reserved. Objectives \u26ab Upon completion of this chapter, you will be able to: \uf070 Describe what cloud computing is. \uf070 Describe the history of cloud computing. \uf070 List a few use cases of cloud comp

# Dataset Splits

In [8]:
dataset = Dataset.from_list(data)

#train (80%), validation (10%), test (10%)
dataset = dataset.train_test_split(test_size=0.2, seed=42)
temp = dataset["test"].train_test_split(test_size=0.5, seed=42)

dataset = {
    "train": dataset["train"],
    "validation": temp["train"],
    "test": temp["test"]
}

print(f"Train: {len(dataset['train'])} examples")
print(f"Validation: {len(dataset['validation'])} examples")
print(f"Test: {len(dataset['test'])} examples")

Train: 6433 examples
Validation: 804 examples
Test: 805 examples


# Format Examples for Training

In [9]:
def format_example(example):
    """Format each example for the model.

    v2 dataset already stores the real instruction used at generation time
    (e.g. "Generate 5 mcq questions" / "Generate 8 mixed questions") in
    example["instruction"] -- use that directly instead of a hardcoded
    "mix mcq and true/false" string. Training on a prompt that doesn't match
    what the sample actually contains would teach the model the wrong
    instruction-to-output mapping.
    """
    prompt = f"""### Instruction
{example['instruction']}.

Rules:
- Provide the correct answer for each question
- Make questions clear and educational
- Base questions strictly on the given input

### Input
{example['input']}

### Response
{example['output']}"""

    return {"text": prompt}

# Apply formatting to all splits
for split in ["train", "validation", "test"]:
    dataset[split] = dataset[split].map(format_example)

print("\nFormatted example:")
print(dataset["train"][0]["text"][:500] + "...")


Map:   0%|          | 0/6433 [00:00<?, ? examples/s]

Map:   0%|          | 0/804 [00:00<?, ? examples/s]

Map:   0%|          | 0/805 [00:00<?, ? examples/s]


Formatted example:
### Instruction
Generate 8 mixed questions.

Rules:
- Provide the correct answer for each question
- Make questions clear and educational
- Base questions strictly on the given input

### Input
The Activation Function Differentiability is the only requirement that an activation function has to satisfy in the BP Algorithm. ◦ This is required to compute the  for each neuron. Sigmoidal functions are commonly used, since they satisfy such a condition: ◦ Logistic Function ◦ Hyperbolic Tangent Functi...


#  Configure Model and Tokenizer

In [10]:
model_name = "mistralai/Mistral-7B-Instruct-v0.2"

# Quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Loading model (this may take a few minutes)...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map={"": 0},
    trust_remote_code=True
)
print("Model loaded successfully!")
print(f"Model device: {next(model.parameters()).device}")

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Model loaded successfully!
Model device: cuda:0


# Configure LoRA

In [11]:
# LoRA configuration
lora_config = LoraConfig(
    r=16,                       # Rank
    lora_alpha=32,              # Scaling factor
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Apply LoRA to model
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 13,631,488 || all params: 7,255,363,584 || trainable%: 0.1879


## Check real token lengths before picking max_length

`max_length=1024` was a guess. v2 lecture chunks are capped around 300 words
at generation time, but quiz outputs (especially `mixed` mode with 8
questions) add meaningfully more tokens on top. Measuring this directly
avoids wasting compute padding every example out to a length most of them
don't need -- padding shorter sequences out to a needlessly long max_length
means the model does real (if useless) compute on padding tokens for every
single step.

In [12]:
import numpy as np

_sample_lengths = [
    len(tokenizer(ex["text"])["input_ids"])
    for ex in dataset["train"].select(range(min(500, len(dataset["train"]))))
]

print(f"Token length over a sample of {len(_sample_lengths)} formatted examples:")
print(f"  min    : {min(_sample_lengths)}")
print(f"  median : {int(np.median(_sample_lengths))}")
print(f"  p90    : {int(np.percentile(_sample_lengths, 90))}")
print(f"  p99    : {int(np.percentile(_sample_lengths, 99))}")
print(f"  max    : {max(_sample_lengths)}")

MAX_LENGTH = int(np.ceil(np.percentile(_sample_lengths, 95) / 64) * 64)
print(f"\nUsing MAX_LENGTH = {MAX_LENGTH} (covers ~95th percentile, rounded to a multiple of 64)")
print("If this is still close to 1024, the savings here will be small -- check the p99/max values above.")


Token length over a sample of 500 formatted examples:
  min    : 262
  median : 608
  p90    : 1012
  p99    : 1760
  max    : 2347

Using MAX_LENGTH = 1280 (covers ~95th percentile, rounded to a multiple of 64)
If this is still close to 1024, the savings here will be small -- check the p99/max values above.


# Tokenize Dataset

In [13]:
def tokenize_function(example):
    """Tokenize examples and create labels"""
    tokenized = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
        return_tensors=None
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized
    
print("Tokenizing datasets...")
train_tokenized = dataset["train"].map(tokenize_function, batched=True, remove_columns=["text", "input", "output"])
val_tokenized = dataset["validation"].map(tokenize_function, batched=True, remove_columns=["text", "input", "output"])
test_tokenized = dataset["test"].map(tokenize_function, batched=True, remove_columns=["text", "input", "output"])

print(f"Tokenized {len(train_tokenized)} training examples")


Tokenizing datasets...


Map:   0%|          | 0/6433 [00:00<?, ? examples/s]

Map:   0%|          | 0/804 [00:00<?, ? examples/s]

Map:   0%|          | 0/805 [00:00<?, ? examples/s]

Tokenized 6433 training examples


# Data Collator

In [14]:
def data_collator(batch):
    """Custom data collator for causal LM"""
    return {
        "input_ids": torch.stack([torch.tensor(f["input_ids"]) for f in batch]),
        "attention_mask": torch.stack([torch.tensor(f["attention_mask"]) for f in batch]),
        "labels": torch.stack([torch.tensor(f["labels"]) for f in batch])
    }

# Training Arguments

In [15]:
training_args = TrainingArguments(
    output_dir="./mistral_quiz_model",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=1,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    fp16=True,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
    remove_unused_columns=False,
    dataloader_pin_memory=False
)


In [16]:
import gc

model.zero_grad(set_to_none=True)
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize() if torch.cuda.is_available() else None

if torch.cuda.is_available():
    print(f"Allocated : {torch.cuda.memory_allocated() / 1024**3:.2f} GiB")
    print(f"Reserved  : {torch.cuda.memory_reserved() / 1024**3:.2f} GiB")


Allocated : 3.89 GiB
Reserved  : 6.28 GiB


In [17]:
import time

model.train()
_sample_batch = data_collator([train_tokenized[i] for i in range(training_args.per_device_train_batch_size)])
_sample_batch = {k: v.to(model.device) for k, v in _sample_batch.items()}

try:
    torch.cuda.synchronize() if torch.cuda.is_available() else None
    _start = time.time()
    _out = model(**_sample_batch)
    _out.loss.backward()
    torch.cuda.synchronize() if torch.cuda.is_available() else None
    _step_time = time.time() - _start

    model.zero_grad()
    torch.cuda.empty_cache()

    steps_per_epoch = len(train_tokenized) // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)
    total_steps = steps_per_epoch * training_args.num_train_epochs
    # gradient_accumulation_steps forward+backward passes happen per optimizer step
    est_seconds = _step_time * training_args.gradient_accumulation_steps * total_steps

    print(f"Measured time for one forward+backward pass : {_step_time:.2f}s")
    print(f"Optimizer steps per epoch                    : {steps_per_epoch}")
    print(f"Total optimizer steps ({training_args.num_train_epochs} epochs)              : {total_steps}")
    print(f"\nEstimated total training time                : {est_seconds/3600:.1f} hours")
    print(f"Estimated time per epoch                     : {est_seconds/training_args.num_train_epochs/3600:.1f} hours")
    print("\n(Rough estimate -- actual time varies with eval passes, checkpointing, and I/O overhead.")
    print(" Treat this as a floor, not an exact prediction.)")

except torch.cuda.OutOfMemoryError as e:
    model.zero_grad(set_to_none=True)
    torch.cuda.empty_cache()
    print("OOM at this batch_size/MAX_LENGTH combination.")
    print(f"  Current per_device_train_batch_size = {training_args.per_device_train_batch_size}")
    print(f"  Current MAX_LENGTH                  = {MAX_LENGTH}")
    print("\nNext step: lower per_device_train_batch_size by half (and double")
    print("gradient_accumulation_steps to keep the same effective batch size),")
    print("or set gradient_checkpointing=True, in the training-arguments cell above,")
    print("then re-run the memory-cleanup cell and this cell again.")


OOM at this batch_size/MAX_LENGTH combination.
  Current per_device_train_batch_size = 1
  Current MAX_LENGTH                  = 1280

Next step: lower per_device_train_batch_size by half (and double
gradient_accumulation_steps to keep the same effective batch size),
or set gradient_checkpointing=True, in the training-arguments cell above,
then re-run the memory-cleanup cell and this cell again.


# Initialize Trainer

In [18]:
# Initialize trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    data_collator=data_collator
)

torch.cuda.empty_cache()
print("Trainer initialized. Ready to train!")

Trainer initialized. Ready to train!


# Train the Model

In [19]:
print("Starting training...")
print("=" * 50)
trainer.train()
print("=" * 50)
print("Training completed!")

Starting training...


Epoch,Training Loss,Validation Loss
1,0.459479,0.476484


Training completed!


# Save the Fine-tuned Model

In [20]:
model = trainer.model

lora_path = "./mistral_quiz_lora"
model.save_pretrained(lora_path)
tokenizer.save_pretrained(lora_path)

print(f"LoRA weights saved to {lora_path}")
print("\nContents:")
!ls -la {lora_path}

import json
with open(f"{lora_path}/adapter_config.json", 'r') as f:
    adapter_config = json.load(f)
print("\nAdapter configuration:")
print(json.dumps(adapter_config, indent=2))

LoRA weights saved to ./mistral_quiz_lora

Contents:
total 56736
drwxr-xr-x 2 root root     4096 Jun 28 08:25 .
drwxr-xr-x 5 root root     4096 Jun 28 08:25 ..
-rw-r--r-- 1 root root     1016 Jun 28 08:25 adapter_config.json
-rw-r--r-- 1 root root 54560368 Jun 28 08:25 adapter_model.safetensors
-rw-r--r-- 1 root root     1058 Jun 28 08:25 chat_template.jinja
-rw-r--r-- 1 root root     5222 Jun 28 08:25 README.md
-rw-r--r-- 1 root root      463 Jun 28 08:25 tokenizer_config.json
-rw-r--r-- 1 root root  3505874 Jun 28 08:25 tokenizer.json

Adapter configuration:
{
  "alora_invocation_tokens": null,
  "alpha_pattern": {},
  "arrow_config": null,
  "auto_mapping": null,
  "base_model_name_or_path": "mistralai/Mistral-7B-Instruct-v0.2",
  "bias": "none",
  "corda_config": null,
  "ensure_weight_tying": false,
  "eva_config": null,
  "exclude_modules": null,
  "fan_in_fan_out": false,
  "inference_mode": true,
  "init_lora_weights": true,
  "layer_replication": null,
  "layers_pattern": null

# Create Complete Model Package for Download

In [21]:
def create_model_package(lora_path, output_path="./mistral_quiz_complete"):
    """Package the model with instructions for loading"""
    
    os.makedirs(output_path, exist_ok=True)
    
    model.save_pretrained(os.path.join(output_path, "lora"))
    tokenizer.save_pretrained(os.path.join(output_path, "lora"))
    
    with open(os.path.join(output_path, "base_model_info.txt"), "w") as f:
        f.write("BASE MODEL: mistralai/Mistral-7B-Instruct-v0.2\n")
        f.write("FINE-TUNING: LoRA\n")
        f.write("TASK: Quiz Generation\n\n")
        f.write("TO LOAD THIS MODEL:\n")
        f.write("```python\n")
        f.write("from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline\n")
        f.write("from peft import PeftModel\n\n")
        f.write("# Load base model\n")
        f.write("base_model = AutoModelForCausalLM.from_pretrained(\n")
        f.write("    'mistralai/Mistral-7B-Instruct-v0.2',\n")
        f.write("    device_map={'': 0},  # pin to one GPU, see notebook top\n")
        f.write("    torch_dtype=torch.float16\n")
        f.write(")\n\n")
        f.write("# Load tokenizer\n")
        f.write("tokenizer = AutoTokenizer.from_pretrained('mistralai/Mistral-7B-Instruct-v0.2')\n\n")
        f.write("# Load LoRA weights\n")
        f.write("model = PeftModel.from_pretrained(base_model, './lora')\n\n")
        f.write("# Create pipeline\n")
        f.write("generator = pipeline('text-generation', model=model, tokenizer=tokenizer)\n")
        f.write("```\n")
    
    with open(os.path.join(output_path, "example_usage.py"), "w") as f:
        f.write("""import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from peft import PeftModel

# Load model
base_model = AutoModelForCausalLM.from_pretrained(
    'mistralai/Mistral-7B-Instruct-v0.2',
    device_map={'': 0},  # pin to one GPU, see notebook top
    torch_dtype=torch.float16
)
tokenizer = AutoTokenizer.from_pretrained('mistralai/Mistral-7B-Instruct-v0.2')
model = PeftModel.from_pretrained(base_model, './lora')

# Create pipeline
generator = pipeline('text-generation', model=model, tokenizer=tokenizer)

# Generate quiz
prompt = \"\"\"### Instruction
Generate computer science quiz questions.

### Input
What is a neural network and how does it learn?

### Response\"\"\"

result = generator(prompt, max_new_tokens=300, do_sample=True, temperature=0.7)[0]['generated_text']
print(result)
""")
    
    return output_path

package_path = create_model_package(lora_path)
print(f"Model package created at: {package_path}")

Model package created at: ./mistral_quiz_complete


# Create Zip File for Download

In [22]:
def create_downloadable_zip(package_path, zip_name="mistral_quiz_model.zip"):
    """Create a zip file of the model package"""
    
    with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(package_path):
            for file in files:
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, os.path.dirname(package_path))
                zipf.write(file_path, arcname)
    
    size_mb = os.path.getsize(zip_name) / (1024 * 1024)
    print(f"Created {zip_name} ({size_mb:.2f} MB)")
    return zip_name

zip_filename = create_downloadable_zip(package_path)

small_zip = "mistral_quiz_adapter_only.zip"
with zipfile.ZipFile(small_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(lora_path):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, os.path.dirname(lora_path))
            zipf.write(file_path, arcname)

small_size = os.path.getsize(small_zip) / (1024 * 1024)
print(f"Created {small_zip} ({small_size:.2f} MB) - This is just the adapter!")

Created mistral_quiz_model.zip (48.65 MB)
Created mistral_quiz_adapter_only.zip (48.65 MB) - This is just the adapter!


# Kaggle-Specific Download Instructions

In [23]:
print("\n" + "="*60)
print("📥 DOWNLOAD INSTRUCTIONS FOR KAGGLE")
print("="*60)

print("""
Option 1: Download via Kaggle UI (Recommended for large files)
---------------------------------------------------------------------------
1. In the Kaggle notebook, look at the right sidebar under "Output"
2. You should see the following files:
   - mistral_quiz_model.zip (complete package with instructions)
   - mistral_quiz_adapter_only.zip (just the LoRA weights - MUCH SMALLER)

3. Click the checkbox next to the file you want
4. Click the "Download" button at the top of the Output section

Option 2: For large files that won't download via browser
---------------------------------------------------------------------------
Run this cell to generate a download script:
""")

download_script = """
# To download via Kaggle API (run locally, not in notebook)
# Install Kaggle API first: pip install kaggle

import os
from kaggle.api.kaggle_api_extended import KaggleApi

# Initialize API
api = KaggleApi()
api.authenticate()

# Download the output files from your notebook
# Replace 'YOUR_USERNAME/YOUR_NOTEBOOK' with your actual notebook path
api.notebook_output_download(
    'YOUR_USERNAME/YOUR_NOTEBOOK',
    path='./downloaded_model'
)

print("Files downloaded to ./downloaded_model")
"""

with open("kaggle_download_instructions.py", "w") as f:
    f.write(download_script)

print("\n✅ Download script saved as 'kaggle_download_instructions.py'")


📥 DOWNLOAD INSTRUCTIONS FOR KAGGLE

Option 1: Download via Kaggle UI (Recommended for large files)
---------------------------------------------------------------------------
1. In the Kaggle notebook, look at the right sidebar under "Output"
2. You should see the following files:
   - mistral_quiz_model.zip (complete package with instructions)
   - mistral_quiz_adapter_only.zip (just the LoRA weights - MUCH SMALLER)

3. Click the checkbox next to the file you want
4. Click the "Download" button at the top of the Output section

Option 2: For large files that won't download via browser
---------------------------------------------------------------------------
Run this cell to generate a download script:


✅ Download script saved as 'kaggle_download_instructions.py'


# Load Evaluation Metrics

In [24]:
class QuizEvaluator:
    """Comprehensive evaluator for quiz generation"""
    
    def __init__(self):
        print("Initializing evaluation metrics...")
        
        # Load metrics
        self.rouge = evaluate.load('rouge')
        self.bleu = evaluate.load('bleu')
        self.bertscorer = BERTScorer(lang="en", rescale_with_baseline=True)
        self.sentence_model = SentenceTransformer('all-MiniLM-L6-v2')
        
        # Load QA model for answerability
        try:
            from transformers import pipeline as qa_pipeline
            self.qa_model = qa_pipeline(
                "question-answering",
                model="distilbert-base-cased-distilled-squad",
                device=0 if torch.cuda.is_available() else -1
            )
            print("✓ QA model loaded")
        except Exception as e:
            self.qa_model = None
            print(f"✗ QA model not available: {e}")
        
        print("✓ Evaluation metrics ready")
    
    def parse_quiz(self, text: str) -> List[Dict]:
        """Parse generated text into structured questions"""
        questions = []
        
        # Pattern for MCQ
        mcq_pattern = r'Q\d+[:.)\s]*([^?\n]+\??)\s*\nA[).]?\s*(.+?)\nB[).]?\s*(.+?)\nC[).]?\s*(.+?)\nD[).]?\s*(.+?)\nAnswer:?\s*([A-D])'
        mcq_matches = re.findall(mcq_pattern, text, re.DOTALL | re.IGNORECASE)
        
        for match in mcq_matches:
            questions.append({
                'type': 'mcq',
                'question': match[0].strip(),
                'options': [match[1].strip(), match[2].strip(), 
                           match[3].strip(), match[4].strip()],
                'answer': match[5].strip()
            })
        
        # Pattern for True/False
        tf_pattern = r'Q\d+[:.)\s]*([^?\n]+\??)\s*\nAnswer:?\s*(True|False)'
        tf_matches = re.findall(tf_pattern, text, re.DOTALL | re.IGNORECASE)
        
        for match in tf_matches:
            questions.append({
                'type': 'tf',
                'question': match[0].strip(),
                'answer': match[1].strip().lower() == 'true'
            })
        
        return questions
    
    def compute_lexical(self, generated: str, reference: str) -> Dict:
        """Compute ROUGE, BLEU, METEOR"""
        results = {}
        
        try:
            # ROUGE
            rouge_scores = self.rouge.compute(predictions=[generated], references=[reference])
            results.update({f"rouge_{k}": v for k, v in rouge_scores.items()})
            
            # BLEU
            bleu = self.bleu.compute(predictions=[generated], references=[[reference]])
            results["bleu"] = bleu['bleu']
            
            # METEOR
            meteor = meteor_score([reference.split()], generated.split())
            results["meteor"] = meteor
            
        except Exception as e:
            print(f"Error in lexical metrics: {e}")
            
        return results
    
    def compute_semantic(self, generated: str, reference: str) -> Dict:
        """Compute BERTScore and SBERT similarity"""
        results = {}
        
        try:
            # BERTScore
            P, R, F1 = self.bertscorer.score([generated], [reference])
            results.update({
                "bertscore_precision": float(P[0].cpu().numpy()),
                "bertscore_recall": float(R[0].cpu().numpy()),
                "bertscore_f1": float(F1[0].cpu().numpy())
            })
            
            # SBERT similarity
            emb1 = self.sentence_model.encode(generated, convert_to_tensor=True)
            emb2 = self.sentence_model.encode(reference, convert_to_tensor=True)
            sim = util.pytorch_cos_sim(emb1, emb2)
            results["sbert_similarity"] = float(sim[0][0].cpu().numpy())
            
        except Exception as e:
            print(f"Error in semantic metrics: {e}")
            
        return results
    
    def evaluate_answerability(self, question: Dict, context: str) -> Dict:
        """Check if question is answerable from context"""
        if not self.qa_model:
            return {'answerability_score': 0.0, 'answer_match': 0.0}
        
        try:
            result = self.qa_model(
                question=question['question'], 
                context=context
            )
            
            # Check if answer matches expected
            expected = str(question.get('answer', ''))
            match = expected.lower() in result['answer'].lower() if expected else False
            
            return {
                'answerability_score': float(result['score']),
                'answer_match': float(match)
            }
        except:
            return {'answerability_score': 0.0, 'answer_match': 0.0}
    
    def evaluate_quality(self, question: str) -> Dict:
        """Evaluate linguistic quality"""
        words = nltk.word_tokenize(question)
        word_count = len(words)
        
        # Metrics
        has_question_mark = 1.0 if '?' in question else 0.0
        conciseness = min(1.0, 20.0 / max(word_count, 1))
        
        question_starts = ['what', 'which', 'who', 'when', 'where', 'why', 'how', 
                          'is', 'are', 'can', 'does', 'do', 'explain', 'define']
        first_word = words[0].lower() if words else ''
        starts_correctly = 1.0 if first_word in question_starts else 0.0
        
        return {
            'word_count': word_count,
            'has_question_mark': has_question_mark,
            'conciseness': conciseness,
            'starts_correctly': starts_correctly,
            'quality_score': (has_question_mark + conciseness + starts_correctly) / 3
        }

# Create Generation Pipeline

In [25]:
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=300,
    do_sample=True,
    temperature=0.7,
    top_p=0.9
)

print("Generation pipeline created")

Passing `generation_config` together with generation-related arguments=({'top_p', 'max_new_tokens', 'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Generation pipeline created


# Run Evaluation

In [26]:
def evaluate_model(evaluator, generator, test_dataset, num_samples=20):
    """Run complete evaluation on test set"""
    
    # Prepare test examples
    test_examples = []
    for item in test_dataset.select(range(min(num_samples, len(test_dataset)))):
        text = item['text']
        instruction_match = re.search(r'### Instruction\n(.*?)\n\nRules:', text, re.DOTALL)
        input_match = re.search(r'### Input\n(.*?)\n\n### Response', text, re.DOTALL)
        response_match = re.search(r'### Response\n(.*?)$', text, re.DOTALL)
        
        if instruction_match and input_match and response_match:
            instruction = instruction_match.group(1).strip()
            test_examples.append({
                'input': input_match.group(1).strip(),
                'reference': response_match.group(1).strip(),
                'prompt': f"### Instruction\n{instruction}.\n\nRules:\n- Provide the correct answer for each question\n- Make questions clear and educational\n- Base questions strictly on the given input\n\n### Input\n{input_match.group(1).strip()}\n\n### Response"
            })
    
    print(f"\n Evaluating on {len(test_examples)} test examples...")
    
    results = {
        'lexical': [],
        'semantic': [],
        'answerability': [],
        'quality': [],
        'per_example': []
    }
    
    for i, example in enumerate(tqdm(test_examples)):
        try:
            # Generate quiz
            generated = generator(example['prompt'])[0]['generated_text']
            
            # Extract just the response part
            if '### Response' in generated:
                generated = generated.split('### Response')[-1].strip()
            
            # Parse questions
            questions = evaluator.parse_quiz(generated)
            
            # Compute metrics
            lexical = evaluator.compute_lexical(generated, example['reference'])
            semantic = evaluator.compute_semantic(generated, example['reference'])
            
            # Store metrics
            results['lexical'].append(lexical)
            results['semantic'].append(semantic)
            
            # Question-level metrics
            for q in questions:
                # Answerability
                ans = evaluator.evaluate_answerability(q, example['input'])
                results['answerability'].append(ans)
                
                # Quality
                qual = evaluator.evaluate_quality(q['question'])
                results['quality'].append(qual)
            
            # Store per-example details
            results['per_example'].append({
                'input': example['input'][:200] + '...',
                'generated': generated,
                'reference': example['reference'][:200] + '...',
                'num_questions': len(questions),
                'lexical_scores': lexical,
                'semantic_scores': semantic
            })
            
        except Exception as e:
            print(f"Error on example {i}: {e}")
    
    return results

# Initialize evaluator
evaluator = QuizEvaluator()

# Run evaluation
print("\n" + "="*60)
print("RUNNING MODEL EVALUATION")
print("="*60)

evaluation_results = evaluate_model(evaluator, generator, dataset["test"], num_samples=20)

Initializing evaluation metrics...


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✓ QA model loaded
✓ Evaluation metrics ready

RUNNING MODEL EVALUATION

📊 Evaluating on 20 test examples...


 10%|█         | 2/20 [00:49<07:18, 24.39s/it]Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
100%|██████████| 20/20 [07:28<00:00, 22.44s/it]


# Compute and Display Final Metrics

In [27]:
def compute_final_metrics(evaluation_results):
    """Compute averages from evaluation results"""
    
    final_metrics = {}
    
    # Lexical averages
    if evaluation_results['lexical']:
        for key in evaluation_results['lexical'][0].keys():
            final_metrics[f'avg_{key}'] = np.mean([m[key] for m in evaluation_results['lexical']])
    
    # Semantic averages
    if evaluation_results['semantic']:
        for key in evaluation_results['semantic'][0].keys():
            final_metrics[f'avg_{key}'] = np.mean([m[key] for m in evaluation_results['semantic']])
    
    # Answerability averages
    if evaluation_results['answerability']:
        for key in evaluation_results['answerability'][0].keys():
            final_metrics[f'avg_{key}'] = np.mean([m[key] for m in evaluation_results['answerability']])
    
    # Quality averages
    if evaluation_results['quality']:
        for key in ['word_count', 'has_question_mark', 'conciseness', 'starts_correctly', 'quality_score']:
            final_metrics[f'avg_{key}'] = np.mean([q[key] for q in evaluation_results['quality']])
    
    # Statistics
    total_questions = len(evaluation_results['quality'])
    final_metrics['total_questions_generated'] = total_questions
    final_metrics['avg_questions_per_example'] = total_questions / len(evaluation_results['per_example'])
    
    return final_metrics

# Compute final metrics
final_metrics = compute_final_metrics(evaluation_results)

# Display results
print("\n" + "="*60)
print(" FINAL EVALUATION RESULTS")
print("="*60)

print("\n LEXICAL METRICS:")
lexical_cols = [k for k in final_metrics.keys() if k.startswith('avg_rouge') or k in ['avg_bleu', 'avg_meteor']]
for k in lexical_cols:
    print(f"  {k}: {final_metrics[k]:.4f}")

print("\n SEMANTIC METRICS:")
semantic_cols = [k for k in final_metrics.keys() if k.startswith('avg_bertscore') or k == 'avg_sbert_similarity']
for k in semantic_cols:
    print(f"  {k}: {final_metrics[k]:.4f}")

print("\n ANSWERABILITY METRICS:")
ans_cols = [k for k in final_metrics.keys() if k.startswith('avg_answerability') or k == 'avg_answer_match']
for k in ans_cols:
    print(f"  {k}: {final_metrics[k]:.4f}")

print("\n QUALITY METRICS:")
print(f"  avg_quality_score: {final_metrics.get('avg_quality_score', 0):.4f}")
print(f"  avg_conciseness: {final_metrics.get('avg_conciseness', 0):.4f}")
print(f"  avg_word_count: {final_metrics.get('avg_word_count', 0):.1f}")

print(f"\n GENERATION STATISTICS:")
print(f"  Total questions: {final_metrics['total_questions_generated']}")
print(f"  Questions per prompt: {final_metrics['avg_questions_per_example']:.2f}")


📊 FINAL EVALUATION RESULTS

📈 LEXICAL METRICS:
  avg_rouge_rouge1: 0.5189
  avg_rouge_rouge2: 0.2235
  avg_rouge_rougeL: 0.3261
  avg_rouge_rougeLsum: 0.4851
  avg_bleu: 0.2082
  avg_meteor: 0.3414

🎯 SEMANTIC METRICS:
  avg_bertscore_precision: 0.5118
  avg_bertscore_recall: 0.4648
  avg_bertscore_f1: 0.4887
  avg_sbert_similarity: 0.8362

✅ ANSWERABILITY METRICS:
  avg_answerability_score: 0.1288
  avg_answer_match: 0.1266

📝 QUALITY METRICS:
  avg_quality_score: 0.5522
  avg_conciseness: 0.9351
  avg_word_count: 18.9

📊 GENERATION STATISTICS:
  Total questions: 79
  Questions per prompt: 3.95


# Save Evaluation Results

In [28]:
# Save results to files
with open('evaluation_results.json', 'w') as f:
    json.dump({
        'summary': final_metrics,
        'detailed': evaluation_results['per_example']
    }, f, indent=2)

# Create readable report
with open('evaluation_report.txt', 'w') as f:
    f.write("QUIZ GENERATION MODEL EVALUATION REPORT\n")
    f.write("="*50 + "\n\n")
    
    f.write("LEXICAL METRICS:\n")
    for k in lexical_cols:
        f.write(f"  {k}: {final_metrics[k]:.4f}\n")
    
    f.write("\nSEMANTIC METRICS:\n")
    for k in semantic_cols:
        f.write(f"  {k}: {final_metrics[k]:.4f}\n")
    
    f.write("\nANSWERABILITY METRICS:\n")
    for k in ans_cols:
        f.write(f"  {k}: {final_metrics[k]:.4f}\n")
    
    f.write("\nQUALITY METRICS:\n")
    f.write(f"  avg_quality_score: {final_metrics.get('avg_quality_score', 0):.4f}\n")
    
    f.write(f"\nSTATISTICS:\n")
    f.write(f"  Total questions: {final_metrics['total_questions_generated']}\n")
    f.write(f"  Questions per prompt: {final_metrics['avg_questions_per_example']:.2f}\n")

print("\n✅ Results saved to:")
print("  - evaluation_results.json")
print("  - evaluation_report.txt")


✅ Results saved to:
  - evaluation_results.json
  - evaluation_report.txt


# Show Sample Generations

In [29]:
print("\n" + "="*60)
print(" SAMPLE GENERATIONS")
print("="*60)

for i, example in enumerate(evaluation_results['per_example'][:3]):
    print(f"\n--- Example {i+1} ---")
    print(f"Input: {example['input'][:150]}...")
    print(f"\nGenerated Quiz:")
    print(example['generated'])
    print(f"\nLexical Scores: ROUGE-1={example['lexical_scores'].get('rouge1', 0):.3f}, "
          f"BLEU={example['lexical_scores'].get('bleu', 0):.3f}")
    print("-" * 40)


📝 SAMPLE GENERATIONS

--- Example 1 ---
Input: At sender: •Divide the original data into the ‘m’ number of blocks with ‘n’ data bits in each block. •Adding all the ‘m’ data blocks. •The addition re...

Generated Quiz:
Q1: In the process of calculating the checksum for a segment, if the receiver's computed checksum matches the checksum field value, it indicates that there was no error detected.
Answer: True

Q2: When adding two 16-bit integers without considering any carry-out, the result is obtained directly by adding the individual digits together.
Answer: False

Q3: The principles of reliable data transfer involve ensuring zero bit errors and no data loss during transmission.
Answer: True

Q4: The function "Reliable Data Transfer (RDT)" assumes an unreliable underlying channel where errors such as bit flips can occur.
Answer: False

Q5: Negative acknowledgements (NAKs) are used by the receiver to inform the sender about errors encountered while processing incoming packets.
Answer: T